# Facility calibration — 02 shunting and parking

Single source of truth for every calibrated facility value, and the generator
for `FACILITY_CALIBRATION.md`. Run `01_source_extraction.ipynb` first.

Scope is what a service facility charges for handling a train that is **not in
commercial service**: shunting movements at a terminal, stabling for the
layover between two nights, and the power the train draws while it stands.
Track access belongs to `models/infrastructure/tac/`, traction energy to
`models/infrastructure/energy_pricing/`, cleaning and maintenance to the
operator side.

Outputs, all generated: `data/facility_charges.csv`,
`data/facility_reference_rotation.csv`, `seed/track_facility.csv`,
`seed/track_facility_default.csv`, `seed/sources.csv`, and
`FACILITY_CALIBRATION.md`.

In [ ]:
# Facility calibration — shunting, stabling and hotel power
#
# Single source of truth for every calibrated value. The CSVs under `data/` and
# `seed/` and the document FACILITY_CALIBRATION.md are generated artifacts:
# re-run this notebook (after 01) to regenerate them, never hand-edit.

import csv
from dataclasses import dataclass, asdict
from pathlib import Path


def _resolve_data_dir() -> Path:
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/facility/calib/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
print(f"data directory: {DATA_DIR}")

CALIBRATION_REVIEWED = "2026-08-17"

# --- provenance vocabulary -------------------------------------------------
SOURCED = "sourced"  # named document, named locator
DERIVED = "derived"  # arithmetic on other values, formula in the note
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
NOT_LEVIED = "not_levied"  # positively documented as absent, not unknown
MISSING = "missing"  # nothing read yet — never an estimate
NO_RAILWAY = "no_railway"

# --- price basis and conversion -------------------------------------------
TARGET_YEAR = 2032

# Two escalation rates, because the two charges have different cost drivers —
# and saying so is the point. Shunting is an hour of a locomotive and two or
# three people: labour dominates it, and Eurostat's labour cost index for
# transportation and storage has run consistently above headline HICP for a
# decade, so it carries HICP plus roughly half a point. Stabling is a length of
# track and the land under it, with no wage signal at all in the sourced rates
# (a low-wage and a high-wage country charge the same 0.15-0.20 EUR/m/day), so
# it carries HICP flat.
SHUNTING_ESCALATION_PER_YEAR = 0.025
PARKING_ESCALATION_PER_YEAR = 0.020
ESCALATION_LOW = 0.015
ESCALATION_HIGH = 0.030

ESCALATION_BY_PARAMETER = {
    "shunting_im": SHUNTING_ESCALATION_PER_YEAR,
    "shunting_allin": SHUNTING_ESCALATION_PER_YEAR,
    "market_topup_loco_crew": SHUNTING_ESCALATION_PER_YEAR,
    "market_topup_loco_only": SHUNTING_ESCALATION_PER_YEAR,
}
"""Everything not named here escalates at the parking rate — the stabling
terms, the hotel-power rate (a metered energy price, so HICP like the traction
price the energy domain carries at the same 2%) and the reference figures."""

# ECB reference rates, snapshot deliberately shared with the TAC and energy
# pricing calibrations so all three infrastructure domains reach EUR on
# identical terms and a scenario repins one date rather than three.
FX_SNAPSHOT = "2026-08-11"
FX_TO_EUR = {
    "EUR": 1.0,
    "CHF": 1.064,
    "CZK": 1 / 24.5,
    "DKK": 1 / 7.46,
    "GBP": 1.20,
    "HUF": 1 / 396.0,
    "NOK": 1 / 11.7,
    "PLN": 1 / 4.25,
    "RON": 1 / 5.08,
    "SEK": 1 / 11.2,
}


def escalation_rate(parameter: str) -> float:
    return ESCALATION_BY_PARAMETER.get(parameter, PARKING_ESCALATION_PER_YEAR)


def to_eur(value: float, currency: str) -> float:
    return value * FX_TO_EUR[currency]


def escalate(value: float, basis_year: int, rate: float) -> float:
    return value * (1.0 + rate) ** (TARGET_YEAR - basis_year)


def to_model_value(
    value: float, currency: str, basis_year: int, parameter: str
) -> float:
    """The number the database receives: EUR, at the evaluation year. Both
    conversions happen exactly once, here."""
    return escalate(to_eur(value, currency), basis_year, escalation_rate(parameter))

## Value record

One row per country and parameter, in the currency and price basis the source
document uses.

In [ ]:
# --- value record ----------------------------------------------------------
@dataclass
class SV:
    """One calibrated value with its provenance. `value` is always as
    published, in `currency` at `basis_year`."""

    country_code: str
    parameter: str
    value: float | None
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: int | None = None
    note: str = ""
    low: float | None = None
    high: float | None = None

    def __post_init__(self):
        if self.status == ASSUMED and (self.low is None or self.high is None):
            raise ValueError(
                f"{self.country_code}.{self.parameter}: ASSUMED needs a band"
            )
        if self.status in (SOURCED, NOT_LEVIED) and not (
            self.source_id and self.locator
        ):
            raise ValueError(
                f"{self.country_code}.{self.parameter}: {self.status} needs a source and a locator"
            )
        if self.status == DERIVED and not self.note:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: derived must state its formula"
            )
        if self.value is not None and self.basis_year is None:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: a value needs a price basis"
            )

    @property
    def eur(self) -> float | None:
        return None if self.value is None else to_eur(self.value, self.currency)

    @property
    def model_value(self) -> float | None:
        if self.value is None:
            return None
        return to_model_value(
            self.value, self.currency, self.basis_year, self.parameter
        )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]

# The reference rotation: one locomotive and ten coaches arriving in the
# morning and leaving the same evening. Two shunting movements and one stabling
# occupation per turnaround, and a turnaround serves both the inbound and the
# outbound trip — so a one-way trip books one shunting event and half a
# parking event at each terminal it uses.
REF_LENGTH_M, REF_HOURS = 300.0, 12.0

COUNTRIES = [
    "AT",
    "BE",
    "BG",
    "CH",
    "CY",
    "CZ",
    "DE",
    "DK",
    "EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IT",
    "LT",
    "LU",
    "LV",
    "MT",
    "NL",
    "NO",
    "PL",
    "PT",
    "RO",
    "SE",
    "SI",
    "SK",
    "UK",
]
NO_RAILWAY_COUNTRIES = {"CY", "MT"}

## Scope classes, the market top-up and the labour index

The one structural decision in this domain: the raw tariffs span a factor of
150 for the same physical movement because they differ in **scope**, not in
price level, so the model adds what the infrastructure manager does *not* sell.

In [ ]:
# --- Scope, top-up, labour index -------------------------------------------
FULL, CREW, TRACK = "full", "crew", "track"
TOPUP = {FULL: 0.0, CREW: 110.0, TRACK: 190.0}
SCOPE_MEANING = {
    FULL: "the IM supplies locomotive and crew",
    CREW: "the IM supplies crew only",
    TRACK: "the IM supplies track and facility access only",
}

# Three tiers from the Ramboll driver-hour band (62-104 EUR/h, a 1.68x spread
# around a midpoint of 83), applied to the market top-up only. This is the ONLY
# per-country variation the seventeen countries without a facility price list
# receive, and it is the one piece of variation with evidence behind it — the
# alternative would be inventing per-country tariffs.
LABOUR_INDEX = {
    **{
        cc: 1.25
        for cc in (
            "AT",
            "BE",
            "CH",
            "DE",
            "DK",
            "FI",
            "FR",
            "IE",
            "LU",
            "NL",
            "NO",
            "SE",
            "UK",
        )
    },
    **{cc: 1.00 for cc in ("CZ", "EE", "ES", "IT", "PT", "SI")},
    **{cc: 0.75 for cc in ("BG", "GR", "HR", "HU", "LT", "LV", "PL", "RO", "SK")},
}
assert set(LABOUR_INDEX) == set(COUNTRIES) - NO_RAILWAY_COUNTRIES

DEFAULT_SHUNT_IM = 10.0
DEFAULT_PARK_RATE = 0.20

BASIS = [
    SV(
        "--",
        "market_topup_loco_crew",
        190.0,
        "EUR/event",
        ASSUMED,
        "NOX-MODEL",
        "line 213",
        "EUR",
        2030,
        "One hour of shunting locomotive plus crew bought on the market. "
        "Calibrated so the German rotation reproduces the nox model's OPS "
        "Infrastructure Access line (307.55 EUR/trip) and cross-checked against "
        "the three full-scope sourced tariffs (AT 99, ES 148, HU 263; mean ~170). "
        "The single biggest assumption in this domain: one real third-party "
        "shunting quotation would materially tighten it.",
        low=120.0,
        high=260.0,
    ),
    SV(
        "--",
        "market_topup_loco_only",
        110.0,
        "EUR/event",
        ASSUMED,
        "NOX-MODEL",
        "line 213",
        "EUR",
        2030,
        "The locomotive alone, where the IM already supplies the crew (GR, PT).",
        low=70.0,
        high=160.0,
    ),
    SV(
        "--",
        "labour_index_band",
        1.0,
        "index",
        ASSUMED,
        "RAMBOLL-BMDV",
        "driver-hour table",
        "EUR",
        2025,
        "Three-tier index anchored on the 62-104 EUR/h driver-hour band. "
        "Applied to the market top-up only, never to the IM tariff or to "
        "stabling.",
        low=0.75,
        high=1.25,
    ),
    SV(
        "--",
        "default_shunting_im",
        DEFAULT_SHUNT_IM,
        "EUR/event",
        ASSUMED,
        "DE-APS-2027",
        "§2.1 Zugbildung I",
        "EUR",
        2027,
        "The track-access-only level observed wherever the IM sells the facility "
        "rather than the service (DE 10.80, SI 13.27, PL 1.72). Used for the "
        "seventeen countries with no facility price list read. Since the top-up "
        "dominates, a real tariff moves the all-in figure by a few per cent.",
        low=0.0,
        high=40.0,
    ),
    SV(
        "--",
        "default_parking_rate",
        DEFAULT_PARK_RATE,
        "EUR/m/day",
        ASSUMED,
        "BG-NRIC-2026",
        "§12",
        "EUR",
        2027,
        "Median of the four sourced length-based tariffs (AT 0.150, BG 0.20, "
        "NO 0.123, HR 0.053). Gives 60 EUR/day for the 300 m reference train, "
        "which lands between the sourced per-event figures (DE 72.24, GR 60.00, "
        "AT 45.10) — that convergence is why it is the default.",
        low=0.05,
        high=0.35,
    ),
    SV(
        "--",
        "hotel_power_eur_hour",
        13.533,
        "EUR/h",
        SOURCED,
        "DE-APS-2027",
        "Elektrant, unmetered flat rate",
        "EUR",
        2027,
        "Power drawn while stabled, as the European proxy. DB InfraGO publishes "
        "both a metered rate (0.23452 EUR/kWh) and this unmetered flat "
        "alternative, and the ratio implies a 57.7 kW standing load — a "
        "plausible draw for a stabled sleeper set on reduced HVAC, which is what "
        "makes the flat figure usable rather than merely convenient. Charged on "
        "actual stabled hours, never on the billable hours after a free track "
        "allowance: the power flows whether or not the siding is free.",
    ),
]
print(f"{len(LABOUR_INDEX)} countries indexed, {len(BASIS)} basis values")

## Sourced tariffs

Eleven countries publish a shunting tariff and eleven a stabling basis. Every
other country takes the European default above.

In [ ]:
# --- Sourced shunting tariffs ----------------------------------------------
SHUNT: dict[str, tuple[str, SV]] = {
    "AT": (
        FULL,
        SV(
            "AT",
            "shunting_im",
            99.25,
            "EUR/event",
            DERIVED,
            "AT-SNNB-2026",
            "Tab.44",
            "EUR",
            2026,
            "39.70 EUR/h for a shunting leader with locomotive operation x 5 h minimum deployment, "
            "shared across the two movements of one turnaround",
        ),
    ),
    "BE": (
        TRACK,
        SV(
            "BE",
            "shunting_im",
            39.004714,
            "EUR/event",
            SOURCED,
            "BE-NS-2027",
            "App F.2 sheet 3.1.1.1",
            "EUR",
            2026,
            "unit cost per access to a service facility",
        ),
    ),
    "BG": (
        TRACK,
        SV(
            "BG",
            "shunting_im",
            2.20,
            "EUR/event",
            SOURCED,
            "BG-NRIC-2026",
            "§2.6.2",
            "EUR",
            2026,
            "0.20 EUR per draw-out/marshalling track use, assumed per vehicle x 11",
        ),
    ),
    "DE": (
        TRACK,
        SV(
            "DE",
            "shunting_im",
            10.80,
            "EUR/event",
            SOURCED,
            "DE-APS-2027",
            "§2.1 Zugbildung I",
            "EUR",
            2027,
            "one hour on a train-formation object; the locomotive and crew are a separate market",
        ),
    ),
    "DK": (
        TRACK,
        SV(
            "DK",
            "shunting_im",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "DK-NS-2027",
            "§3.5",
            "EUR",
            2027,
            "no infrastructure charges for operations or parking on sidings",
        ),
    ),
    "ES": (
        FULL,
        SV(
            "ES",
            "shunting_im",
            148.00,
            "EUR/event",
            SOURCED,
            "ES-ADIF-2027",
            "ch.6 basic operations",
            "EUR",
            2027,
            "shunting driving operations 148 EUR/h; overall shunting operations 200 EUR/h",
        ),
    ),
    "GR": (
        CREW,
        SV(
            "GR",
            "shunting_im",
            30.00,
            "EUR/event",
            SOURCED,
            "GR-OSE-2026",
            "§6.4.1",
            "EUR",
            2019,
            "per manoeuvre, passenger; covers the manoeuvring team but not the locomotive or driver",
        ),
    ),
    "HU": (
        FULL,
        SV(
            "HU",
            "shunting_im",
            263.00,
            "EUR/event",
            DERIVED,
            "HU-NS-2627",
            "Annex 5.2-6",
            "EUR",
            2027,
            "24,480 HUF/person/h staff + 79,752 HUF/vehicle/h traction unit for one hour, converted at "
            "the pinned snapshot — the only IM that prices the shunting locomotive itself",
        ),
    ),
    "PL": (
        TRACK,
        SV(
            "PL",
            "shunting_im",
            1.72,
            "EUR/event",
            ASSUMED,
            "PL-PLK-A91",
            "Annex 9.1",
            "EUR",
            2027,
            "3.66 PLN/km under electric traction x an assumed 2 km per movement. Actual distances are "
            "tabulated per location in Appendix 2.8; the all-in total is under 2% sensitive to this.",
            low=0.9,
            high=4.6,
        ),
    ),
    "PT": (
        CREW,
        SV(
            "PT",
            "shunting_im",
            24.59,
            "EUR/event",
            SOURCED,
            "PT-IP-2027",
            "§5.4.4",
            "EUR",
            2027,
            "long duration above 30 min; short duration is 9.75 EUR",
        ),
    ),
    "SI": (
        TRACK,
        SV(
            "SI",
            "shunting_im",
            13.27,
            "EUR/event",
            SOURCED,
            "SI-NS-2027",
            "§5.4.2 P22",
            "EUR",
            2027,
            "per departure from and arrival at the station of origin or a yard",
        ),
    ),
}

# --- Sourced stabling bases ------------------------------------------------
# Four bases exist in Europe and the model carries all four, because reducing
# them to one would misprice the two that are structurally different: Germany
# charges by the hour and is length-independent by design, and three countries
# price a stabling operation as an event with no time term at all.
PER_METRE_DAY, PER_HOUR, PER_EVENT, NONE = (
    "per_metre_day",
    "per_hour",
    "per_event",
    "none",
)

PARK: dict[str, dict] = {
    "AT": dict(
        rate=SV(
            "AT",
            "parking_rate",
            0.150,
            "EUR/m/day",
            SOURCED,
            "AT-SNNB-2026",
            "Tab.49 item 4.2.2",
            "EUR",
            2026,
            "4.57 EUR/m/month on a long-term booking, the realistic mode for a daily rotation",
        )
    ),
    "BG": dict(
        rate=SV(
            "BG",
            "parking_rate",
            0.20,
            "EUR/m/day",
            SOURCED,
            "BG-NRIC-2026",
            "§12",
            "EUR",
            2026,
            "per metre of length per 24 h",
        )
    ),
    "NO": dict(
        free_h=48.0,
        rate=SV(
            "NO",
            "parking_rate",
            0.123,
            "EUR/m/day",
            SOURCED,
            "NO-NS-2027",
            "Tab.9 §7.3.5",
            "EUR",
            2026,
            "6 NOK/h per commenced 100 m outside Alnabru",
        ),
    ),
    "HR": dict(
        free_h=24.0,
        rate=SV(
            "HR",
            "parking_rate",
            0.053,
            "EUR/m/day",
            SOURCED,
            "HR-NS-2027",
            "§7.3.4.4 note 9",
            "EUR",
            2027,
            "the above-48 h band, establishment category 3",
        ),
    ),
    "DE": dict(
        hour=SV(
            "DE",
            "parking_per_hour",
            6.02,
            "EUR/h",
            SOURCED,
            "DE-APS-2027",
            "§2.1 Abstellung I",
            "EUR",
            2027,
            "per started use-hour, length-independent by design — the APS calls the charge expressly "
            "zuglaengenunabhaengig. Abstellung II is 2.68 and III 1.74",
        )
    ),
    "PT": dict(
        free_h=1.0,
        hour=SV(
            "PT",
            "parking_per_hour",
            2.43,
            "EUR/h",
            DERIVED,
            "PT-IP-2027",
            "§5.4.4",
            "EUR",
            2027,
            "Te = 0.0405 EUR/min x 60 min; applies beyond the first hour, and timetabled technical "
            "stops are exempt",
        ),
    ),
    "IT": dict(
        event=SV(
            "IT",
            "parking_per_event",
            9.992,
            "EUR/event",
            SOURCED,
            "IT-NS-2027",
            "§5.4.6.1",
            "EUR",
            2027,
            "indirect cost per parking operation; the energy element of the same section belongs to "
            "the energy pricing domain",
        )
    ),
    "GR": dict(
        event=SV(
            "GR",
            "parking_per_event",
            60.00,
            "EUR/event",
            DERIVED,
            "GR-OSE-2026",
            "§6.4.2",
            "EUR",
            2019,
            "stabling is defined as exactly 2 manoeuvres x 30 EUR — no time and no length term",
        )
    ),
    "DK": dict(
        event=SV(
            "DK",
            "parking_per_event",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "DK-NS-2027",
            "§3.5",
            "EUR",
            2027,
            "no siding charges",
        )
    ),
    "SI": dict(
        event=SV(
            "SI",
            "parking_per_event",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "SI-NS-2027",
            "§5.4.3 P23",
            "EUR",
            2027,
            "planned storage is not charged; P23 covers unplanned RU-attributable storage only",
        )
    ),
    "BE": dict(
        event=SV(
            "BE",
            "parking_per_event",
            0.0,
            "EUR/event",
            NOT_LEVIED,
            "BE-NS-2027",
            "App F.2 sheet 3.1.2.2",
            "EUR",
            2026,
            "occupancy is charged only in yards declared congested",
        )
    ),
}
print(f"{len(SHUNT)} sourced shunting tariffs, {len(PARK)} sourced stabling bases")

## Assembly

The all-in shunting figure and the stabling basis per country, both carried to
the evaluation year on their own escalation rate.

In [ ]:
# --- Assembly --------------------------------------------------------------
values: list[SV] = list(BASIS)
rotation: list[dict] = []

HOTEL_POWER = next(v for v in BASIS if v.parameter == "hotel_power_eur_hour")


def _default_shunt_im(cc: str) -> SV:
    return SV(
        cc,
        "shunting_im",
        DEFAULT_SHUNT_IM,
        "EUR/event",
        ASSUMED,
        "DE-APS-2027",
        "§2.1 Zugbildung I",
        "EUR",
        2027,
        "European default — no facility price list read for this country, see "
        "default_shunting_im",
        low=0.0,
        high=40.0,
    )


def _default_park_rate(cc: str) -> SV:
    return SV(
        cc,
        "parking_rate",
        DEFAULT_PARK_RATE,
        "EUR/m/day",
        ASSUMED,
        "BG-NRIC-2026",
        "§12",
        "EUR",
        2027,
        "European default — see default_parking_rate",
        low=0.05,
        high=0.35,
    )


def _reference_parking_eur(basis: str, rate: SV, free_h: float) -> float:
    """What the reference rotation pays for stabling, in EUR at the evaluation
    year — the same arithmetic calc_facility.py applies at runtime, kept here
    so the document can show a comparable figure per country."""
    billable = max(0.0, REF_HOURS - free_h)
    if basis == NONE or billable <= 0.0 and basis != PER_EVENT:
        return 0.0
    if basis == PER_EVENT:
        return rate.model_value
    if basis == PER_HOUR:
        return rate.model_value * -(-billable // 1)  # started hours
    days = max(1.0, -(-billable // 24))  # started 24 h periods
    return rate.model_value * REF_LENGTH_M * days


for cc in sorted(LABOUR_INDEX):
    scope, im = SHUNT.get(cc, (TRACK, _default_shunt_im(cc)))
    index = LABOUR_INDEX[cc]
    topup = TOPUP[scope] * index
    allin = SV(
        cc,
        "shunting_allin",
        round(im.value + topup, 4),
        "EUR/event",
        DERIVED,
        im.source_id,
        im.locator,
        im.currency if im.currency == "EUR" else "EUR",
        im.basis_year,
        f"IM tariff {to_eur(im.value, im.currency):.2f} EUR + {scope} top-up "
        f"{TOPUP[scope]:.0f} x labour index {index:.2f}",
    )
    values.append(im)
    values.append(allin)

    spec = PARK.get(cc, {})
    free_h = spec.get("free_h", 0.0)
    if "event" in spec:
        basis, rate = PER_EVENT, spec["event"]
    elif "hour" in spec:
        basis, rate = PER_HOUR, spec["hour"]
    else:
        basis, rate = PER_METRE_DAY, spec.get("rate") or _default_park_rate(cc)
    if rate.value == 0.0:
        basis = NONE
    values.append(rate)

    park_ref = _reference_parking_eur(basis, rate, free_h)
    hotel_ref = HOTEL_POWER.model_value * REF_HOURS
    values.append(
        SV(
            cc,
            "parking_per_event_ref",
            round(park_ref, 2),
            "EUR/event",
            DERIVED,
            rate.source_id,
            rate.locator,
            "EUR",
            TARGET_YEAR,
            f"{basis} basis at {REF_LENGTH_M:.0f} m and {REF_HOURS:.0f} h, "
            f"{free_h:.0f} h free allowance, EUR at {TARGET_YEAR}",
        )
    )

    rotation.append(
        dict(
            country_code=cc,
            scope=scope,
            labour_index=index,
            shunting_im_eur_event=round(im.eur, 4),
            shunting_allin_eur_event=round(allin.model_value, 2),
            parking_basis=basis,
            parking_rate=round(rate.model_value, 6),
            parking_free_hours=free_h,
            parking_eur_event_ref=round(park_ref, 2),
            hotel_power_eur_event_ref=round(hotel_ref, 2),
            turnaround_eur=round(2 * allin.model_value + park_ref + hotel_ref, 2),
        )
    )

# A shunting movement anywhere in Europe is an hour of a locomotive and a crew;
# a figure outside this band means the scope classification or the top-up broke.
for row in rotation:
    assert 80.0 <= row["shunting_allin_eur_event"] <= 400.0, (
        f"{row['country_code']}: shunting all-in {row['shunting_allin_eur_event']} implausible"
    )

_allin = sorted(r["shunting_allin_eur_event"] for r in rotation)
print(
    f"shunting all-in EUR/event at {TARGET_YEAR}: "
    f"{_allin[0]:.0f} to {_allin[-1]:.0f}, spread {_allin[-1] / _allin[0]:.1f}x"
)
_park = sorted(r["parking_eur_event_ref"] for r in rotation)
print(f"parking EUR/event (300 m, 12 h): {_park[0]:.0f} to {_park[-1]:.0f}")
print(f"hotel power EUR/event: {HOTEL_POWER.model_value * REF_HOURS:.2f}")

## Cross-check against the operator model

The nox model is the only operator-side source that separates infrastructure
access from cleaning and servicing, which makes it the one available validation
of the market top-up.

In [ ]:
# --- Cross-check -----------------------------------------------------------
# nox line 213 is an INFRASTRUCTURE ACCESS figure: shunting plus stabling. Hotel
# power is deliberately excluded from the comparison — nox books servicing
# separately at 2,599 EUR/trip, which is where standing power plausibly sits,
# and folding it in here would silently break the only cross-check available.
NOX_PER_TRIP = 307.55
NOX_PER_TRIP_YEAR = 2030

_de = next(r for r in rotation if r["country_code"] == "DE")
_access = 2 * _de["shunting_allin_eur_event"] + _de["parking_eur_event_ref"]
_per_trip = _access / 2
_nox_2032 = escalate(NOX_PER_TRIP, NOX_PER_TRIP_YEAR, PARKING_ESCALATION_PER_YEAR)
_im_only = (2 * _de["shunting_im_eur_event"] + _de["parking_eur_event_ref"]) / 2

print(f"DE access per one-way trip : {_per_trip:7.2f} EUR ({TARGET_YEAR})")
print(
    f"nox line 213               : {_nox_2032:7.2f} EUR (carried from {NOX_PER_TRIP_YEAR})"
)
print(f"deviation                  : {_per_trip / _nox_2032 - 1:+7.1%}")
print(
    f"IM tariffs alone           : {_im_only:7.2f} EUR "
    f"({_im_only / _nox_2032:.0%} of the operator figure)"
)
print(
    f"hotel power on top         : {_de['hotel_power_eur_event_ref'] / 2:7.2f} EUR/trip"
)

assert abs(_per_trip / _nox_2032 - 1) < 0.25, (
    "the top-up calibration has drifted from the operator model by more than 25% — "
    "re-derive it rather than widening this assert"
)

## Export

In [ ]:
# --- Export ----------------------------------------------------------------


def write_csv(path: Path, columns: list[str], rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {path.name}: {len(rows)} rows")


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    write_csv(DATA_DIR / name, columns, rows)


# Fill the grid: every railway country carries every parameter, so a NULL is a
# statement rather than an absence.
DB_PARAMETERS = ["shunting_allin", "parking_rate", "parking_per_event_ref"]
calibrated = {(v.country_code, v.parameter): v for v in values}
full_grid = list(values)
for cc in NO_RAILWAY_COUNTRIES:
    for parameter in DB_PARAMETERS:
        full_grid.append(
            SV(cc, parameter, None, "", NO_RAILWAY, note="No railway network")
        )

write_data("facility_charges.csv", SV_FIELDS, [asdict(v) for v in full_grid])
write_data("facility_reference_rotation.csv", list(rotation[0]), rotation)

by_status: dict[str, int] = {}
for v in full_grid:
    by_status[v.status] = by_status.get(v.status, 0) + 1
print()
for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
    print(f"    {s:11} {n:4}")

_register_path = DATA_DIR / "sources_register.csv"
with open(_register_path, encoding="utf-8") as fh:
    register = {r["source_id"]: r for r in csv.DictReader(fh)}
cited = {v.source_id for v in full_grid if v.source_id}
assert cited <= set(register), (
    f"cited but not in the register: {sorted(cited - set(register))}"
)
print(f"\n  provenance: {len(cited)} sources cited, all present in the register")

## Seed export

One row per country, EUR at the evaluation year. The four stabling bases travel
as a mode plus one rate: `calc_facility.py` reads the mode and applies the
matching arithmetic, so a country that prices per hour and one that prices per
metre never share a column.

In [ ]:
# --- Seed export -----------------------------------------------------------
SEED_DIR = DATA_DIR.parent / "seed"
SEED_DIR.mkdir(exist_ok=True)

TRACK_FACILITY_COLUMNS = [
    "country_code",
    "track_shunting_eur_event",
    "track_parking_basis",
    "track_parking_eur_metre_day",
    "track_parking_eur_hour",
    "track_parking_eur_event",
    "track_parking_free_hours",
    "track_parking_hotel_power_eur_hour",
    "track_parking_eur_day",
    "source_id",
    "change_log",
]

_SEED_NDIGITS = 6


def _fmt(value: float | None) -> str:
    if value is None:
        return ""
    return f"{value:.{_SEED_NDIGITS}f}".rstrip("0").rstrip(".")


_BASIS_COLUMN = {
    PER_METRE_DAY: "track_parking_eur_metre_day",
    PER_HOUR: "track_parking_eur_hour",
    PER_EVENT: "track_parking_eur_event",
}

facility_seed = []
for row in rotation:
    cc = row["country_code"]
    # cc_cited, not cited: the module-level `cited` set is what the source
    # export below writes, and shadowing it here would reduce the seeded
    # register to whichever country happened to sort last.
    cc_cited = sorted(
        {
            v.source_id
            for v in values
            if v.country_code == cc and v.source_id and v.value is not None
        }
    )
    primary = cc_cited[0] if cc_cited else "DE-APS-2027"
    extra = (
        f"Facility charges also sourced from "
        f"{', '.join(s for s in cc_cited if s != primary)}."
        if len(cc_cited) > 1
        else ""
    )
    seed_row = {
        "country_code": cc,
        "track_shunting_eur_event": _fmt(row["shunting_allin_eur_event"]),
        "track_parking_basis": row["parking_basis"],
        "track_parking_eur_metre_day": "",
        "track_parking_eur_hour": "",
        "track_parking_eur_event": "",
        "track_parking_free_hours": _fmt(row["parking_free_hours"]),
        "track_parking_hotel_power_eur_hour": _fmt(HOTEL_POWER.model_value),
        # Display only: what the reference rotation pays for one stabling
        # occupation, so the API keeps one headline figure per country the way
        # it keeps a flat track access rate. The cost model never reads it.
        "track_parking_eur_day": _fmt(row["parking_eur_event_ref"]),
        "source_id": primary,
        "change_log": extra,
    }
    column = _BASIS_COLUMN.get(row["parking_basis"])
    if column is not None:
        seed_row[column] = _fmt(row["parking_rate"])
    facility_seed.append(seed_row)

# The fallback row. A country the model routes through but the calibration has
# no facility figures for gets the European default directly: the track-scope
# shunting default at index 1.00 and the median stabling rate. Unlike the TAC
# and energy groups there is no median-of-calibrated step, because the default
# IS the calibration for seventeen of the twenty-eight countries already — a
# median over them would just re-derive it with extra steps.
_default_shunt = to_model_value(
    DEFAULT_SHUNT_IM + TOPUP[TRACK] * 1.00, "EUR", 2027, "shunting_allin"
)
_default_rate = to_model_value(DEFAULT_PARK_RATE, "EUR", 2027, "parking_rate")
default_facility = {
    "country_code": "_default",
    "track_shunting_eur_event": _fmt(_default_shunt),
    "track_parking_basis": PER_METRE_DAY,
    "track_parking_eur_metre_day": _fmt(_default_rate),
    "track_parking_free_hours": _fmt(0.0),
    "track_parking_hotel_power_eur_hour": _fmt(HOTEL_POWER.model_value),
    "track_parking_eur_day": _fmt(_default_rate * REF_LENGTH_M),
}
print(
    f"  default shunting {_default_shunt:.2f} EUR/event, "
    f"stabling {_default_rate:.4f} EUR/m/day"
)

write_csv(
    SEED_DIR / "track_facility_default.csv", TRACK_FACILITY_COLUMNS, [default_facility]
)
write_csv(SEED_DIR / "track_facility.csv", TRACK_FACILITY_COLUMNS, facility_seed)

SOURCE_SEED_COLUMNS = ["source_id", "source_description", "source_url", "source_date"]
source_seed = [
    {
        "source_id": sid,
        "source_description": (
            f"{register[sid]['title']} — {register[sid]['publisher']} "
            f"({register[sid]['pub_year']})"
        ),
        "source_url": register[sid]["url_or_file"],
        "source_date": register[sid]["date_accessed"],
    }
    for sid in sorted(cited)
]
write_csv(SEED_DIR / "sources.csv", SOURCE_SEED_COLUMNS, source_seed)

assert all(r["track_shunting_eur_event"] for r in facility_seed)
assert all(
    r["track_parking_basis"] == NONE or r[_BASIS_COLUMN[r["track_parking_basis"]]]
    for r in facility_seed
), "a country carries a stabling basis with no rate in the matching column"

## Document generation

In [ ]:
# --- FACILITY_CALIBRATION.md generation -------------------------------------
from datetime import date

DOC_PATH = DATA_DIR.parent / "FACILITY_CALIBRATION.md"


def _f(v: float | None, nd: int = 2) -> str:
    if v is None:
        return "—"
    if abs(v) < 0.01:
        return f"{v:.4f}".rstrip("0").rstrip(".")
    return f"{v:,.{nd}f}"


_BASIS_LABEL = {
    PER_METRE_DAY: "per metre per started 24 h",
    PER_HOUR: "per started hour",
    PER_EVENT: "per stabling operation",
    NONE: "not levied",
}

ROTATION_ROWS = "\n".join(
    f"| {r['country_code']} | {r['scope']} | {r['labour_index']:.2f} | "
    f"{_f(r['shunting_im_eur_event'])} | **{_f(r['shunting_allin_eur_event'])}** | "
    f"{_BASIS_LABEL[r['parking_basis']]} | {_f(r['parking_rate'], 4)} | "
    f"{r['parking_free_hours']:.0f} | {_f(r['parking_eur_event_ref'])} | "
    f"**{_f(r['turnaround_eur'])}** |"
    for r in rotation
)

_sourced_shunt = sorted(SHUNT)
_sourced_park = sorted(PARK)
_default_shunt_countries = sorted(set(LABOUR_INDEX) - set(SHUNT))
_default_park_countries = sorted(cc for cc in LABOUR_INDEX if cc not in PARK)

_basis_counts: dict[int, int] = {}
for v in values:
    if v.value is not None and v.basis_year is not None and v.basis_year != TARGET_YEAR:
        _basis_counts[v.basis_year] = _basis_counts.get(v.basis_year, 0) + 1
ESCALATION_ROWS = "\n".join(
    f"| {by} | {n} | {TARGET_YEAR - by} | "
    f"{(1 + PARKING_ESCALATION_PER_YEAR) ** (TARGET_YEAR - by):.3f} | "
    f"{(1 + SHUNTING_ESCALATION_PER_YEAR) ** (TARGET_YEAR - by):.3f} |"
    for by, n in sorted(_basis_counts.items())
)

FX_ROWS = "\n".join(
    f"| {cur} | {1 / FX_TO_EUR[cur]:,.3f} {cur} = 1 EUR | {FX_TO_EUR[cur]:.6f} |"
    for cur in sorted({v.currency for v in values if v.currency != "EUR" and v.value})
)
if not FX_ROWS:
    FX_ROWS = (
        "| — | every sourced tariff is published in euro, or was converted "
        "to euro in the source document itself | — |"
    )

_METHOD_KINDS = {
    "operator_model",
    "study",
    "fx_reference",
    "macro_projection",
    "official_statistics",
}


def _source_row(sid: str) -> str:
    r = register[sid]
    link = r["url_or_file"]
    shown = f"[link]({link})" if link.startswith("http") else f"`{link}`"
    return (
        f"| `{sid}` | {r['title']} | {r['publisher']} | {r['pub_year']} | "
        f"{r['price_basis_year'] or '—'} | {shown} |"
    )


SOURCE_ROWS = "\n".join(
    _source_row(s) for s in sorted(cited) if register[s]["kind"] not in _METHOD_KINDS
)
METHOD_ROWS = "\n".join(
    _source_row(s) for s in sorted(cited) if register[s]["kind"] in _METHOD_KINDS
)
UNCHECKED_ROWS = "\n".join(
    _source_row(s) for s in sorted(register) if register[s]["used"] == "Not used"
)

_status_counts = {
    s: sum(1 for v in full_grid if v.status == s)
    for s in (SOURCED, DERIVED, ASSUMED, NOT_LEVIED, MISSING, NO_RAILWAY)
}
print("document inputs assembled:", {k: v for k, v in _status_counts.items() if v})

In [ ]:
BODY = r"""
## What belongs here

What a **service facility** charges for handling a train that is not in
commercial service:

- **Shunting** — one assisted movement of a trainset or locomotive within a
  facility or between platform and siding, with shunting staff and a shunting
  locomotive.
- **Stabling** — one continuous occupation of a siding during the layover
  between two nights of service.
- **Hotel power** — the electricity the train draws while it stands there.

Not here: track access (`models/infrastructure/tac/`), traction energy while
driving (`models/infrastructure/energy_pricing/`), station and platform stop
charges, cleaning and interior servicing, maintenance-facility use.

### The reference rotation

One locomotive and ten coaches, @@REF_LENGTH_M@@ m, arriving in the morning and
leaving the same evening: **2 shunting movements + 1 stabling occupation of
@@REF_HOURS@@ h per turnaround**. A turnaround serves both the inbound and the
outbound trip, so **one one-way trip books 1 shunting event and half a parking
event at each terminal it uses**. The route builder emits exactly this — two
`Shunting` events per terminal (one from each trip that ends or starts there)
and one `Parking` per terminal, deduplicated by stop.

---

## Why the model adds to the published tariff

The raw tariffs span a factor of **150** for what is physically the same
operation: Hungary charges 263 EUR per movement, Poland 1.72, Denmark nothing.
Reading that as a price-level difference would be wrong. The spread is
**scope**, not cost:

- MÁV's tariff bundles the shunting locomotive itself (79,752 HUF per vehicle-hour) plus crew.
- DB InfraGO charges for the **track only** — Zugbildung I is 10.80 EUR per use-hour, and the locomotive and crew are bought on a separate market.
- Adif prices the whole operation as a service, 148 EUR/h.
- OSE charges 30 EUR per manoeuvre and states explicitly that this covers the manoeuvring team and *not* the locomotive or driver.

So the model prices what the infrastructure manager does not sell:

```
shunting_event = IM_tariff(country) + market_topup(scope) × labour_index(country)
```

| Scope | Meaning | Top-up at index 1.00 | Countries |
|---|---|---|---|
| `full` | @@FULL_MEANING@@ | 0 EUR | @@FULL_COUNTRIES@@ |
| `crew` | @@CREW_MEANING@@ | 110 EUR | @@CREW_COUNTRIES@@ |
| `track` | @@TRACK_MEANING@@ | 190 EUR | the remaining @@N_TRACK@@ |

Applied across Europe this collapses the 150× raw spread to
**@@SPREAD@@× (@@MIN_SHUNT@@–@@MAX_SHUNT@@ EUR)**, which is what one hour of a
shunting locomotive plus crew should cost anywhere on the continent.

### How much of this is actually sourced — read this before trusting a country figure

| | sourced or derived | European default |
|---|---|---|
| Shunting IM tariff | @@N_SHUNT_SOURCED@@ countries | @@N_SHUNT_DEFAULT@@ |
| Stabling basis | @@N_PARK_SOURCED@@ countries | @@N_PARK_DEFAULT@@ |
| Scope class | 5 inferred from tariff wording | 23 assumed `track` |

The seventeen default countries all carry the **same** 10 EUR IM tariff and the
same 0.20 EUR/m/day stabling rate. Their figures differ from one another only
through the labour index, and that is deliberate: the index is the one piece of
per-country variation with evidence behind it (Ramboll prices a driver-hour at
62–104 EUR/h across fifteen countries, a 1.68× spread), where inventing
per-country tariffs would not be. **Read a default country's number as "the
European average, tier-adjusted", not as a national tariff.**

The good news is that this matters less than it looks: the top-up dominates
every all-in figure, so a real price list for a default country moves its total
by single-digit per cents. The countries worth reading a price list for are the
ones a chosen route set actually touches.

---

## Stabling: four bases, all carried

Four regimes exist in Europe and the model keeps all four rather than reducing
them to one, because two of them are structurally different rather than merely
differently priced:

| Basis | Arithmetic | Countries |
|---|---|---|
| `per_metre_day` | `length_m × rate × ceil(billable_h / 24)` | @@PER_METRE_COUNTRIES@@ |
| `per_hour` | `rate × ceil(billable_h)` — length-independent | @@PER_HOUR_COUNTRIES@@ |
| `per_event` | one flat charge per occupation, no time term | @@PER_EVENT_COUNTRIES@@ |
| `none` | positively documented as not levied | @@NONE_COUNTRIES@@ |

`billable_h` is the stabled hours minus a country's free allowance —
@@FREE_COUNTRIES@@ — and a free allowance long enough to cover the layover
zeroes the charge entirely, which is why Norway and Croatia cost nothing on a
12 h turnaround.

**Germany is length-independent by design.** The Anlagenpreissystem charges per
started use-hour per usage object and calls the disponent charge expressly
*"zuglängenunabhängig"*. That is why a per-metre model cannot simply be
extended over it.

### The German disponent factor, deliberately not modelled

APS 2027 prices facilities with an *Anlagendisponent* by a factor rising 1→10
over the first ten hours and constant at 10 thereafter. A 12 h stay on such a
facility costs **722 EUR** rather than 72; a 24 h stay 1,445 EUR. Whether a
given facility is disponent-managed is published per facility in the Liste der
Serviceeinrichtungen, and a route does not know which siding it will be given —
so the model prices the plain Abstellung I rate and this stays an open item. It
is the largest single unpriced risk in this domain: an order of magnitude on
one country's stabling line.

---

## Hotel power

Power drawn while stabled is charged on **actual stabled hours**, never on the
billable hours after a free track allowance: the electricity flows whether or
not the siding is free.

The rate is @@HOTEL_RATE@@ EUR/h, applied in every country as the European
proxy. It is DB InfraGO's unmetered Elektrant flat rate, and it validates
itself: divided by the metered alternative of 0.23452 EUR/kWh it implies a
**57.7 kW** standing load, which is a plausible draw for a stabled sleeper set
on reduced HVAC. On the reference rotation it adds
@@HOTEL_PER_EVENT@@ EUR per turnaround.

**Pre-heating stays a documented gap.** APS prices 16.7 Hz train pre-heating at
157.828 EUR/h flat — implying 428 kW, and 316 EUR for a two-hour winter
conditioning. It is real money but seasonal, and the model carries no season.

---

## Two conversions, and two escalation rates

Currency conversion uses ECB reference rates at the **@@FX_SNAPSHOT@@**
snapshot, shared with the TAC and energy pricing calibrations so all three
infrastructure domains reach EUR on identical terms:

| Currency | Reference rate | → EUR factor |
|---|---|---|
@@FX_ROWS@@

Price basis is carried to @@TARGET_YEAR@@ at **two different rates**, because
the two charges have different cost drivers:

- **Shunting @@SHUNT_PCT@@/yr.** An hour of a locomotive and two or three
  people. Eurostat's labour cost index for transportation and storage has run
  consistently above headline HICP for a decade, so a labour-dominated charge
  carries HICP plus roughly half a point.
- **Stabling and hotel power @@PARK_PCT@@/yr.** Track and the land under it,
  with no wage signal at all in the sourced rates — Bulgaria charges 0.20
  EUR/m/day and Austria 0.15, a low-wage and a high-wage country on the same
  rate. HICP flat. Hotel power is a metered energy price and carries the same
  2% the energy domain applies to traction electricity.

| Price basis | Values | Years to @@TARGET_YEAR@@ | at @@PARK_PCT@@ | at @@SHUNT_PCT@@ |
|---|---|---|---|---|
@@ESCALATION_ROWS@@

COUNTER-ARGUMENT, recorded: half a point of divergence over five to thirteen
years is a 3–7% difference on the shunting line, which is inside the ±35% band
the market top-up itself carries. The split is defensible but it is not where
the uncertainty in this domain lives.

---

## Master table

Every figure EUR at @@TARGET_YEAR@@. The turnaround column is 2 shunting events
plus one stabling occupation plus @@REF_HOURS@@ h of hotel power, for the
@@REF_LENGTH_M@@ m reference train.

| Country | Scope | Idx | IM €/ev | **Shunting all-in €/ev** | Stabling basis | Rate | Free h | Stabling €/ev | **Turnaround €** |
|---|---|---|---|---|---|---|---|---|---|
@@ROTATION_ROWS@@

---

## Cross-check against the operator model

The nox night-train business case isolates line 213, *OPS Infrastructure
Access*, at **307.55 EUR/trip** in 2030 money — separately from line 212 Track
& Station Access and line 214 Servicing. It is the only operator-side figure
that separates shunting and stabling from cleaning, which is what makes it the
anchor for the market top-up.

| | EUR per one-way trip |
|---|---|
| This calibration, Germany (shunting + stabling) | **@@DE_ACCESS@@** |
| nox line 213, carried to @@TARGET_YEAR@@ | @@NOX_2032@@ |
| deviation | **@@NOX_DEVIATION@@** |
| IM tariffs alone, no top-up | @@IM_ONLY@@ (@@IM_ONLY_SHARE@@ of the operator figure) |

That last line is the justification for the whole top-up construction: pricing
the German rotation from published infrastructure tariffs alone books about a
quarter of what the operator-side model does.

Hotel power is **excluded** from this comparison on purpose — nox books
servicing separately, and adding @@HOTEL_PER_TRIP@@ EUR/trip here would push the
figure @@HOTEL_BREAKS@@ above nox and silently destroy the only validation
available. It is seeded as its own column for exactly that reason.

**Ramboll cross-check, for completeness.** Its *Reinigung/Abstellung* block is
14% of 44.51 EUR/train-km, i.e. ~6.2 EUR/train-km — but it bundles cleaning
with stabling and cannot be split, and the comparable nox pair (servicing 2.63
+ infrastructure access 0.31) is roughly half that. The two operator models
differ by about a factor of two, almost entirely in cleaning rather than in
facility access.

---

## Materiality and open items

For the reference rotation the all-in facility cost is roughly
@@MIN_TURNAROUND@@–@@MAX_TURNAROUND@@ EUR per turnaround against a daily track
access charge of a few thousand — **single-digit to low-teens per cent of
infrastructure cost**. Higher than published IM tariffs alone would suggest,
and high enough to matter for route ranking, but not a line that decides a
business case.

| # | Item | Impact if it moves |
|---|---|---|
| 1 | The 190 / 110 EUR market top-ups are the single biggest assumption here, calibrated to one operator model and cross-checked against three sourced full-scope tariffs | ±35% on the shunting line, i.e. most of this domain |
| 2 | German disponent facilities price stabling 10× on a 12 h stay; which facility a route gets is unknown | an order of magnitude on DE stabling |
| 3 | @@N_SHUNT_DEFAULT@@ countries have no facility price list read | a few per cent each — the top-up dominates |
| 4 | Scope class is inferred from tariff wording for five countries and assumed `track` for the rest | −190 EUR per event for any country that turns out to bundle a locomotive |
| 5 | Pre-heating energy is unpriced (157.828 EUR/h, ~316 EUR per winter conditioning) | seasonal; the model carries no season |
| 6 | Facility-operator versus IM boundary: a zero in the IM column means the *infrastructure manager* charges nothing, not that the operator pays nothing | in CH, DK, DE, NL and UK the stabling facility is typically the incumbent's depot arm, and that bilateral access is the dominant real cost here |

---

## Sources

| source_id | Document | Publisher | Published | Price basis | Link |
|---|---|---|---|---|---|
@@SOURCE_ROWS@@

Cross-check and method sources — these do not price one country, they underpin
the top-up, the labour index and the escalation rates:

| source_id | Document | Publisher | Published | Price basis | Link |
|---|---|---|---|---|---|
@@METHOD_ROWS@@

Registered but not yet read — the documents behind open item 3:

| source_id | Document | Publisher | Published | Price basis | Link |
|---|---|---|---|---|---|
@@UNCHECKED_ROWS@@
"""

In [ ]:
CALIBRATION_TEMPLATE = r"""# Facility Charges — Calibration

Shunting, stabling and hotel power at service facilities, calibrated per
country for @@N_COUNTRIES@@ European countries.

Generated by `02_facility_calibration.ipynb` on @@GENDATE@@ — do not edit by
hand; re-run the notebooks (01 then 02, top to bottom) to regenerate this
document and the CSVs under `data/` and `seed/`. Calibration last reviewed end
to end @@REVIEWED@@. Implementation:
`backend/models/infrastructure/facility/calc_facility.py`.

**Provenance at a glance:** @@N_SOURCED@@ values read from a named locator,
@@N_DERIVED@@ derived by documented arithmetic, @@N_ASSUMED@@ assumed with a
band, @@N_NOT_LEVIED@@ documented as not levied, across @@N_CITED@@ cited
sources. This is the most assumption-heavy of the three infrastructure
domains, and the section "How much of this is actually sourced" says exactly
where the assumptions are.

---

@@BODY@@
"""

In [ ]:
# --- render ----------------------------------------------------------------
TOKENS = {
    "GENDATE": date.today().isoformat(),
    "REVIEWED": CALIBRATION_REVIEWED,
    "TARGET_YEAR": str(TARGET_YEAR),
    "N_COUNTRIES": str(len(rotation)),
    "N_SOURCED": str(_status_counts[SOURCED]),
    "N_DERIVED": str(_status_counts[DERIVED]),
    "N_ASSUMED": str(_status_counts[ASSUMED]),
    "N_NOT_LEVIED": str(_status_counts[NOT_LEVIED]),
    "N_CITED": str(len(cited)),
    "REF_LENGTH_M": f"{REF_LENGTH_M:.0f}",
    "REF_HOURS": f"{REF_HOURS:.0f}",
    "FULL_MEANING": SCOPE_MEANING[FULL],
    "CREW_MEANING": SCOPE_MEANING[CREW],
    "TRACK_MEANING": SCOPE_MEANING[TRACK],
    "FULL_COUNTRIES": ", ".join(
        cc for cc, (s, _) in sorted(SHUNT.items()) if s == FULL
    ),
    "CREW_COUNTRIES": ", ".join(
        cc for cc, (s, _) in sorted(SHUNT.items()) if s == CREW
    ),
    "N_TRACK": str(sum(1 for r in rotation if r["scope"] == TRACK)),
    "SPREAD": f"{_allin[-1] / _allin[0]:.1f}",
    "MIN_SHUNT": _f(_allin[0]),
    "MAX_SHUNT": _f(_allin[-1]),
    "N_SHUNT_SOURCED": str(len(SHUNT)),
    "N_SHUNT_DEFAULT": str(len(_default_shunt_countries)),
    "N_PARK_SOURCED": str(len(PARK)),
    "N_PARK_DEFAULT": str(len(_default_park_countries)),
    "PER_METRE_COUNTRIES": f"{sum(1 for r in rotation if r['parking_basis'] == PER_METRE_DAY)} "
    "countries, four of them on a sourced rate (AT, BG, HR, NO)",
    "PER_HOUR_COUNTRIES": ", ".join(
        r["country_code"] for r in rotation if r["parking_basis"] == PER_HOUR
    ),
    "PER_EVENT_COUNTRIES": ", ".join(
        r["country_code"] for r in rotation if r["parking_basis"] == PER_EVENT
    ),
    "NONE_COUNTRIES": ", ".join(
        r["country_code"] for r in rotation if r["parking_basis"] == NONE
    )
    or "none",
    "FREE_COUNTRIES": ", ".join(
        f"{r['country_code']} {r['parking_free_hours']:.0f} h"
        for r in rotation
        if r["parking_free_hours"] > 0
    ),
    "HOTEL_RATE": _f(HOTEL_POWER.model_value),
    "HOTEL_PER_EVENT": _f(HOTEL_POWER.model_value * REF_HOURS),
    "HOTEL_PER_TRIP": _f(HOTEL_POWER.model_value * REF_HOURS / 2),
    "HOTEL_BREAKS": f"{(_per_trip + HOTEL_POWER.model_value * REF_HOURS / 2) / _nox_2032 - 1:+.0%}",
    "FX_SNAPSHOT": FX_SNAPSHOT,
    "FX_ROWS": FX_ROWS,
    "SHUNT_PCT": f"{SHUNTING_ESCALATION_PER_YEAR:.1%}",
    "PARK_PCT": f"{PARKING_ESCALATION_PER_YEAR:.1%}",
    "ESCALATION_ROWS": ESCALATION_ROWS,
    "ROTATION_ROWS": ROTATION_ROWS,
    "DE_ACCESS": _f(_per_trip),
    "NOX_2032": _f(_nox_2032),
    "NOX_DEVIATION": f"{_per_trip / _nox_2032 - 1:+.1%}",
    "IM_ONLY": _f(_im_only),
    "IM_ONLY_SHARE": f"{_im_only / _nox_2032:.0%}",
    "MIN_TURNAROUND": _f(min(r["turnaround_eur"] for r in rotation), 0),
    "MAX_TURNAROUND": _f(max(r["turnaround_eur"] for r in rotation), 0),
    "SOURCE_ROWS": SOURCE_ROWS,
    "METHOD_ROWS": METHOD_ROWS,
    "UNCHECKED_ROWS": UNCHECKED_ROWS,
    "BODY": BODY.strip(),
}


def _render(text: str) -> str:
    for _ in range(3):
        for key, value in TOKENS.items():
            text = text.replace(f"@@{key}@@", value)
        if "@@" not in text:
            break
    return text


document = _render(CALIBRATION_TEMPLATE)
_left = sorted({t.split("@@")[0] for t in document.split("@@")[1::2]})
assert not _left, f"unsubstituted tokens: {_left}"

DOC_PATH.write_text(document, encoding="utf-8")
print(
    f"  {DOC_PATH.name}: {len(document.splitlines())} lines, {len(document):,} characters"
)

## Display

Imports pandas, which is what keeps this cell out of `db/dev/seed.py`'s
stdlib-only execution path.

In [ ]:
# Display only — pandas is fine here, seed.py skips this cell.
import pandas as pd

pd.DataFrame(rotation).set_index("country_code")